In [ ]:
# Setup tools to avoid package conflicts

!pip install -U pip setuptools

In [ ]:
# Install all necassary packages from shell file

!bash install_packages.sh

In [ ]:
# Just checks devices available

import accelerate as acc
import wandb as wb
import torch
import transformers
import sentencepiece
import numpy as np
import google.protobuf  # Should import without error
print(np.__version__)  # Should print 1.26.4
print(torch.__version__)  # Should print 2.2.1
print(google.protobuf.__version__)  # Should print 4.25.3
print(torch.cuda.is_available())  # True
print(torch.cuda.device_count())  # 2
print(acc.__version__)
print(wb.__version__)

In [ ]:
# Hugging face authentication

from huggingface_hub import login
# Replace with your actual token from huggingface.co/settings/tokens
login(token="YOUR_HF_TOKEN")

In [ ]:
# Configure logging
# Imports
import logging
import os

# Configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
)
logger = logging.getLogger(__name__)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# Functiont to print trainable parameters
def print_trainable_parameters(model):
    """Prints number and percentage of trainable parameters in the model."""
    trainable_params = 0
    all_params = 0
    
    for _, param in model.named_parameters():
        all_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    percentage = 100 * trainable_params / all_params
    logger.info(f"Trainable params: {trainable_params:,} / {all_params:,} "
                f"({percentage:.2f}% of total)")
    return trainable_params, all_params

In [ ]:
# Analyze dataset examples length (tonen check)

from datasets import load_dataset
import matplotlib.pyplot as plt  # ✅ right

MAX_SEQ_LENGTH = 8192

def estimate_tokens(text):
    """Rough estimation of tokens (approx 4 characters per token)"""
    if not text:
        return 0
    return len(text) // 4

train_dataset = load_dataset("json", data_files="train_codeLlama.jsonl", split="train") 

def format_completion_example(example):
    prompt = example.get("prompt", "")
    completion = example.get("completion", "")
    return {"text": f"{prompt}{completion}"}
    
train_dataset = train_dataset.map(format_completion_example, batched=False)
token_counts = [estimate_tokens(example['text']) for example in train_dataset]

# Plot token length distribution
plt.figure(figsize=(10, 5))
plt.hist(token_counts, bins=40, color='skyblue', edgecolor='black')
plt.axvline(x=MAX_SEQ_LENGTH, color='red', linestyle='--', label=f'MAX_SEQ_LENGTH = {MAX_SEQ_LENGTH}')
plt.title('🧮 Token Length Distribution in Training Dataset')
plt.xlabel('Estimated Token Count (approx)')
plt.ylabel('Number of Examples')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Stats
avg_length = np.mean(token_counts)
over_limit = sum(c > MAX_SEQ_LENGTH for c in token_counts)

print(f"📏 Average example length: {avg_length:.2f} tokens")
print(f"⚠️  Examples longer than {MAX_SEQ_LENGTH} tokens: {over_limit} out of {len(train_dataset)}")

In [ ]:
# Complete Training Script using SFT trainer

# Step 1: Necassary Imports
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, EarlyStoppingCallback
from trl import SFTTrainer
from datasets import load_dataset

# Step 2: Verify CUDA and Set base Model
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}, GPUs: {torch.cuda.device_count()}")
BASE_MODEL = "codellama/CodeLlama-7b-hf" # base model codellama

# Step 3: Load dataset
dataset = load_dataset("json", data_files="train_codeLlama.jsonl", split="train") 

# Step 4: Preprocess dataset and split
def format_completion_example(example):
    prompt = example.get("prompt", "")
    completion = example.get("completion", "")
    return {"text": f"{prompt}{completion}"}
    
dataset = dataset.map(format_completion_example, batched=False)

dataset = dataset.map(
    format_completion_example, 
    batched=True,  # Process in batches for better efficiency
    num_proc=4,    # Use multiple processes
    remove_columns=dataset.column_names,
    load_from_cache_file=False  # Disable if you're modifying the processing
)

split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

eval_dataset

# Display raw sample
#print("\n🔎 Sample from raw dataset:")
#print(dataset[0])

#print("\n🔎 Sample from train and eval datasets:")
#print(train_dataset[0])
#print(eval_dataset[0])

# Custom callback for early stopping and overfitting detection
class CustomEarlyStoppingCallback(EarlyStoppingCallback):
    def __init__(self, early_stopping_patience=3, overfitting_threshold=0.1):
        super().__init__(early_stopping_patience=early_stopping_patience)
        self.overfitting_threshold = overfitting_threshold
        self.train_losses = []
        self.eval_losses = []
    
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        # Store losses for analysis
        self.eval_losses.append(metrics.get("eval_loss", float("inf")))
        
        # Check for overfitting (train loss much lower than eval loss)
        if len(self.train_losses) > 0 and len(self.eval_losses) > 0:
            loss_diff = self.eval_losses[-1] - self.train_losses[-1]
            if loss_diff > self.overfitting_threshold:
                print(f"⚠️ Overfitting detected (train loss: {self.train_losses[-1]:.4f}, eval loss: {self.eval_losses[-1]:.4f})")
                control.should_training_stop = True
        
        # Call parent's early stopping logic
        super().on_evaluate(args, state, control, metrics, **kwargs)
    
    def on_step_end(self, args, state, control, **kwargs):
    # Record training loss periodically if available
        if state.log_history:
            last_log = state.log_history[-1]
            loss = last_log.get("loss")
            if loss is not None:
                self.train_losses.append(loss)
        return super().on_step_end(args, state, control, **kwargs)

    
# Step 5: Set 4-bit quantization config (QLORA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16  # BF16 compute for H100
)

# Step 6: Load model
logger.info(f"🧠 Loading model {BASE_MODEL} with 4-bit quantization config")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    #torch_dtype=torch.bfloat16  # Required for H100, aligns with bnb_config
)

# Step 7: Load tokenizer
logger.info(f"📦 Loading tokenizer for model: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Step 8: Prepare model for 4-bit training
model = prepare_model_for_kbit_training(model)

# Step 9: Set LoRA config and get PEFT model for training
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# Enable gradient checkpointing
model.gradient_checkpointing_enable()
# Enable input gradients
model.enable_input_require_grads()

# ✅ Log trainable params
print_trainable_parameters(model)
logger.info("✅ Model ready for LoRA training with 4-bit quantization on H100")

# Step 10: Set training arguments
training_args = TrainingArguments(
    output_dir = f"./results/codellama_SFTTrain",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 2,
    num_train_epochs = 10, # Higher no of epochs for better training
    learning_rate = 2e-4,
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    warmup_ratio = 0.03,
    max_grad_norm=0.3,
    evaluation_strategy = "epoch", # Evaluate at end of each epoch
    save_strategy = "epoch", # Save at end of each epoch
    logging_steps = 10,
    save_total_limit = 3,  # Keep only 3 best checkpoints
    bf16 = True, 
    logging_dir = os.path.join(f"./results/codellama_SFTTrain", "logs"),
    report_to="wandb",  # or "wandb" if using Weights & Biases
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_32bit",
    group_by_length=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False
)

# Step 11: Set up trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    dataset_text_field="text",
    tokenizer = tokenizer,
    args=training_args,
    max_seq_length = 8192,
    callbacks=[
        CustomEarlyStoppingCallback(
            early_stopping_patience=3,  # Stop if no improvement for 3 epochs
            overfitting_threshold=0.15  # Difference threshold between train/eval loss
        )
    ]
)

# Step 12: Train the model
train_result = trainer.train()

# Step 13: Save the best model and tokenizer
best_model_path = os.path.join(training_args.output_dir, "best_model")
trainer.save_model(best_model_path)  # This will save the best model found
tokenizer.save_pretrained(best_model_path)

with open("./results/codellama_SFTTrain/best_model/training_config.txt", "w") as f:
    f.write(str(training_args))

with open("./results/codellama_SFTTrain/best_model/lora_config.txt", "w") as f:
    f.write(str(lora_config))

# Save training metrics
trainer.save_metrics("train", train_result.metrics)